In [1]:
import polars as pl
import pandas as pd
import seaborn as sb
import numpy as np
from pathlib import Path
from datetime import date

In [2]:
# Carregando os dados de penalidades aplicadas a operadoras com scan (LazyFrame - não materializa o DataFrame imediatamente)
lf_origem_penalidades_aplicadas_a_operadoras = pl.scan_parquet(r"c:\GIT\DadosANS\Cargas\processamentos\penalidades_20260511_213501\ingestao\downloads\penalidades_aplicadas_a_operadoras.parquet")

# collect_schema() é usado para obter o esquema do LazyFrame sem materializar os dados; 
# O método .columns() é mais custoso em recursos, pois pode exigir a leitura de uma amostra dos dados para inferir os tipos de coluna.
# O método .names() retorna apenas os nomes das colunas consultando o schema, o que é mais leve.
lf_origem_penalidades_aplicadas_a_operadoras.collect_schema().names()

['NR_DEMANDA',
 'NR_PROCESSO',
 'TIPO_PROCESSO',
 'OBJETO',
 'REGISTRO_OPERADORA',
 'CNPJ',
 'RAZAO_SOCIAL',
 'SITUACAO_OPERADORA',
 'STATUS_DEMANDA',
 'DT_DECISAO_1A',
 'DT_PUBLICACAO_1A',
 'DT_PUBLICACAO_1A_FINAL',
 'VL_MULTA_FIXA_1A',
 'VL_MULTA_DIARIA_1A',
 'VL_TOTAL_APLICADO_1A',
 'DT_CIENCIA_RECONSIDERA_TOTAL',
 'DT_DECISAO_2A',
 'DT_PUBLICACAO_2A',
 'TIPO_DECISAO_2A',
 'VL_MULTA_FIXA_2A',
 'VL_MULTA_DIARIA_2A',
 'VL_TOTAL_APLICADO_2A',
 'VL_MULTA_FINAL_APLICADA',
 'VL_TOTAL_DESCONTOS',
 'DT_ARQUIVAMENTO',
 'MOTIVO_ARQUIVAMENTO',
 'TIPO_PENALIDADE',
 'DT_SUSPENSAO_ADM',
 'DT_SUSPENSAO_JUD',
 'NR_GRU',
 'DT_VENC_GRU',
 'VL_GRU',
 'DE_SITUACAO_GRU',
 'DT_PAGTO_A_VISTA_ANS',
 'VL_PAGO_A_VISTA_ANS',
 'DT_VENC_1A_PARC_ANS',
 'VL_PARCELAS_ANS_PAGAS',
 'STATUS_PARCELAMENTO',
 'DT_INSCRICAO',
 'INSCRITO_DA',
 'ORIGEM_PAGAMENTO',
 'NR_COMPETENCIA_CARGA']

In [3]:
# Definição das colunas de possível interesse para a análise
colunas_desejaveis = [
    'NR_DEMANDA',
    'NR_PROCESSO',
    'TIPO_PROCESSO',
    'REGISTRO_OPERADORA',
    'STATUS_DEMANDA',
    'DT_PUBLICACAO_1A_FINAL',
    'VL_MULTA_FIXA_1A',
    'VL_TOTAL_APLICADO_1A',
    'DT_CIENCIA_RECONSIDERA_TOTAL',
    'TIPO_DECISAO_2A',
    'VL_MULTA_FIXA_2A',
    'VL_TOTAL_APLICADO_2A',
    'VL_MULTA_FINAL_APLICADA',
    'VL_TOTAL_DESCONTOS',
    'DT_ARQUIVAMENTO',
    'MOTIVO_ARQUIVAMENTO',
    'TIPO_PENALIDADE',
    'DT_SUSPENSAO_ADM',
    'DT_SUSPENSAO_JUD',
    'DT_VENC_GRU',
    'VL_GRU',
    'DE_SITUACAO_GRU',
    'DT_PAGTO_A_VISTA_ANS',
    'VL_PAGO_A_VISTA_ANS',
    'DT_INSCRICAO',
    'INSCRITO_DA',
    'ORIGEM_PAGAMENTO',
    'NR_COMPETENCIA_CARGA',
]
# Definição de variável geral para o registro da operadora específica (SC - 312924)
SC = 312924

# Filtragem dos dados para a operadora específica (registro 312924 - SC)
# Seleção das colunas de interesse
# Materializando o DataFrame com collect() - pois já filtramos e selecionamos as colunas, então o resultado deve ser mais leve para a memória
penalidades_sc = (
    lf_origem_penalidades_aplicadas_a_operadoras
    .filter(pl.col("REGISTRO_OPERADORA") == SC)
    .select(colunas_desejaveis)
    .collect()
)


In [4]:
# "CAST" ou conversão das colunas de data tipadas como "str" para o tipo "Date" do Polars, usando o formato específico de data presente nos dados ("%d/%m/%Y").
penalidades_sc = penalidades_sc.with_columns(pl.col(['DT_PUBLICACAO_1A_FINAL',
                                                    'DT_CIENCIA_RECONSIDERA_TOTAL',
                                                    'DT_SUSPENSAO_ADM',
                                                    'DT_SUSPENSAO_JUD',
                                                    'DT_VENC_GRU',
                                                    'DT_PAGTO_A_VISTA_ANS',
                                                    'DT_INSCRICAO',
                                                    'DT_ARQUIVAMENTO']).str.strptime(pl.Date, "%d/%m/%Y", exact=False, strict=False))

# Confirmando a conversão
penalidades_sc.show()

NR_DEMANDA,NR_PROCESSO,TIPO_PROCESSO,REGISTRO_OPERADORA,STATUS_DEMANDA,DT_PUBLICACAO_1A_FINAL,VL_MULTA_FIXA_1A,VL_TOTAL_APLICADO_1A,DT_CIENCIA_RECONSIDERA_TOTAL,TIPO_DECISAO_2A,VL_MULTA_FIXA_2A,VL_TOTAL_APLICADO_2A,VL_MULTA_FINAL_APLICADA,VL_TOTAL_DESCONTOS,DT_ARQUIVAMENTO,MOTIVO_ARQUIVAMENTO,TIPO_PENALIDADE,DT_SUSPENSAO_ADM,DT_SUSPENSAO_JUD,DT_VENC_GRU,VL_GRU,DE_SITUACAO_GRU,DT_PAGTO_A_VISTA_ANS,VL_PAGO_A_VISTA_ANS,DT_INSCRICAO,INSCRITO_DA,ORIGEM_PAGAMENTO,NR_COMPETENCIA_CARGA
i64,str,str,i64,str,date,i64,i64,date,str,i64,i64,i64,i64,date,str,str,date,date,date,i64,str,date,i64,date,str,str,i64
5379468,"""33910.004280/2022-35""","""Consumidor""",312924,"""Arquivado""",2023-06-25,88000,88000,null,null,0,0,88000,17600,2024-02-27,"""Decisão condenatória com pagam…","""Multa Pecuniária""",null,null,2023-08-16,70400,"""Pago""",2023-08-16,70400,null,"""NAO""","""ANS""",202503
3419328,"""33903.003604/2017-40""","""Consumidor""",312924,"""Arquivado""",2017-05-23,66000,66000,null,"""Provimento Negado""",0,0,0,null,2020-09-24,"""Extinção do Processo (Artigo 5…","""Multa Pecuniária""",null,2019-03-15,2019-02-28,73240,"""Cancelada""",null,null,null,"""NAO""","""ANS""",202503
1664105,"""25785.014384/2012-31""","""Consumidor""",312924,"""Em Cobrança""",2015-08-11,72650,72650,null,"""Provimento Negado""",72650,72650,72650,null,null,null,"""Multa Pecuniária""",null,2017-05-17,2017-01-31,99916,"""Suspenso""",null,null,2017-04-03,"""SIM""","""DA""",202503
2580639,"""25782.009138/2015-30""","""Consumidor""",312924,"""Arquivado""",2016-11-03,66000,66000,null,"""Provimento Negado""",66000,66000,66000,null,2022-10-27,"""Decisão condenatória com pagam…","""Multa Pecuniária""",null,null,2019-02-28,90625,"""Pago""",2020-10-06,99239,null,"""NAO""","""ANS""",202503
4431631,"""33910.026096/2019-41""","""Consumidor""",312924,"""Arquivado""",2022-04-06,79200,79200,null,null,0,0,79200,15840,2024-02-20,"""Decisão condenatória com pagam…","""Multa Pecuniária""",null,null,2023-02-10,63360,"""Pago""",2023-02-10,63360,null,"""NAO""","""ANS""",202503


In [5]:
# Análise exploratória: total de multas pago a vista por ano de pagamento
consulta_valor_ano = penalidades_sc.group_by(pl.col("DT_PAGTO_A_VISTA_ANS")
                                             .dt
                                             .year()
                                             .alias("ANO_PAGTO")
                                             ).agg(pl.col("VL_PAGO_A_VISTA_ANS")
                                                                     .sum()
                                                                     .alias("TOTAL_ANO_PAGTO")
                                                                     ).sort("ANO_PAGTO")

consulta_valor_ano.show(14)


ANO_PAGTO,TOTAL_ANO_PAGTO
i32,i64
null,0
2013,126041
2014,170918
2015,185878
2016,711519
2017,995493
2018,4436404
2019,3991809
2020,1966389


# As informações geradas sugerem uma diminuição maior que 50% das multas pagas entre 2024 e 2025, é necessário investigar.

In [6]:
penalidades_sc.filter(pl.col("DT_PAGTO_A_VISTA_ANS").dt.year() >= 2024).sort("DT_PAGTO_A_VISTA_ANS").show(limit=None)

NR_DEMANDA,NR_PROCESSO,TIPO_PROCESSO,REGISTRO_OPERADORA,STATUS_DEMANDA,DT_PUBLICACAO_1A_FINAL,VL_MULTA_FIXA_1A,VL_TOTAL_APLICADO_1A,DT_CIENCIA_RECONSIDERA_TOTAL,TIPO_DECISAO_2A,VL_MULTA_FIXA_2A,VL_TOTAL_APLICADO_2A,VL_MULTA_FINAL_APLICADA,VL_TOTAL_DESCONTOS,DT_ARQUIVAMENTO,MOTIVO_ARQUIVAMENTO,TIPO_PENALIDADE,DT_SUSPENSAO_ADM,DT_SUSPENSAO_JUD,DT_VENC_GRU,VL_GRU,DE_SITUACAO_GRU,DT_PAGTO_A_VISTA_ANS,VL_PAGO_A_VISTA_ANS,DT_INSCRICAO,INSCRITO_DA,ORIGEM_PAGAMENTO,NR_COMPETENCIA_CARGA
i64,str,str,i64,str,date,i64,i64,date,str,i64,i64,i64,i64,date,str,str,date,date,date,i64,str,date,i64,date,str,str,i64
5729924,"""33910.010793/2023-66""","""Consumidor""",312924,"""Arquivado""",2023-09-18,33000,33000,null,null,0,0,33000,6600,2024-04-05,"""Decisão condenatória com pagam…","""Multa Pecuniária""",null,null,2024-01-15,26400,"""Pago""",2024-01-11,26400,null,"""NAO""","""ANS""",202503
5707875,"""33910.006662/2023-84""","""Consumidor""",312924,"""Arquivado""",2023-09-05,66000,66000,null,null,0,0,66000,13200,2024-04-05,"""Decisão condenatória com pagam…","""Multa Pecuniária""",null,null,2024-01-15,52800,"""Pago""",2024-01-11,52800,null,"""NAO""","""ANS""",202503
5764359,"""33910.013807/2023-01""","""Consumidor""",312924,"""Arquivado""",2023-11-09,30000,30000,null,null,0,0,30000,6000,2024-01-25,"""Decisão condenatória com pagam…","""Multa Pecuniária""",null,null,2024-01-17,24000,"""Pago""",2024-01-16,24000,null,"""NAO""","""ANS""",202503
5751718,"""33910.011975/2023-54""","""Consumidor""",312924,"""Arquivado""",2023-09-14,80000,80000,null,null,0,0,80000,32000,2024-01-22,"""Decisão condenatória com pagam…","""Multa Pecuniária""",null,null,2024-01-18,48000,"""Pago""",2024-01-16,48000,null,"""NAO""","""ANS""",202503
5185000,"""33910.032791/2021-66""","""Consumidor""",312924,"""Arquivado""",2023-08-24,60000,60000,null,null,0,0,60000,12000,2024-06-12,"""Decisão condenatória com pagam…","""Multa Pecuniária""",null,null,2024-01-17,48000,"""Pago""",2024-01-17,48000,null,"""NAO""","""ANS""",202503
5428684,"""33910.021611/2022-00""","""Consumidor""",312924,"""Arquivado""",2023-05-15,79200,79200,null,"""Provimento Negado""",79200,79200,79200,null,2024-03-20,"""Decisão condenatória com pagam…","""Multa Pecuniária""",null,null,2024-01-31,84736,"""Pago""",2024-01-31,84736,null,"""NAO""","""ANS""",202503
5579433,"""33910.033562/2022-40""","""Consumidor""",312924,"""Arquivado""",2023-12-28,60000,60000,null,null,0,0,60000,24000,2024-03-04,"""Decisão condenatória com pagam…","""Multa Pecuniária""",null,null,2024-02-14,36000,"""Pago""",2024-02-09,36000,null,"""NAO""","""ANS""",202503
5896923,"""33910.036179/2023-24""","""Consumidor""",312924,"""Arquivado""",2024-01-11,30000,30000,null,null,0,0,30000,12000,2024-03-25,"""Decisão condenatória com pagam…","""Multa Pecuniária""",null,null,2024-02-20,18000,"""Pago""",2024-02-20,18000,null,"""NAO""","""ANS""",202503
4986073,"""33910.006580/2021-78""","""Consumidor""",312924,"""Arquivado""",2023-12-06,88000,88000,null,null,0,0,88000,17600,2024-04-11,"""Decisão condenatória com pagam…","""Multa Pecuniária""",null,null,2024-02-21,70400,"""Pago""",2024-02-20,70400,null,"""NAO""","""ANS""",202503


In [7]:
penalidades_sc.null_count()

NR_DEMANDA,NR_PROCESSO,TIPO_PROCESSO,REGISTRO_OPERADORA,STATUS_DEMANDA,DT_PUBLICACAO_1A_FINAL,VL_MULTA_FIXA_1A,VL_TOTAL_APLICADO_1A,DT_CIENCIA_RECONSIDERA_TOTAL,TIPO_DECISAO_2A,VL_MULTA_FIXA_2A,VL_TOTAL_APLICADO_2A,VL_MULTA_FINAL_APLICADA,VL_TOTAL_DESCONTOS,DT_ARQUIVAMENTO,MOTIVO_ARQUIVAMENTO,TIPO_PENALIDADE,DT_SUSPENSAO_ADM,DT_SUSPENSAO_JUD,DT_VENC_GRU,VL_GRU,DE_SITUACAO_GRU,DT_PAGTO_A_VISTA_ANS,VL_PAGO_A_VISTA_ANS,DT_INSCRICAO,INSCRITO_DA,ORIGEM_PAGAMENTO,NR_COMPETENCIA_CARGA
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,384,248,0,0,0,215,59,59,0,384,351,44,44,44,83,83,353,0,44,0


# A última data de pagamento para o ano de 2025 foi registrada em 10/03/2025

1. Será necessário estimar os valores do ano de 2025 com base no primeiro bimestre do ano e anos anteriores;
2. Descartar Regressão Linear - o histórico de pagamento de multas de anos anteriores demonstra uma relação não linear entre eles;
3. Descartar Regressão Polinomial - a variação positiva e negativa geraria uma equação com muitas curvas, tornando o modelo desnecessariamente complexo;
4. Investigar correlação entre número de reclamações e valores de multa pagos.

In [ ]:
'''
1. select() - seleciona apenas as colunas relevantes para a análise de correlação;

2. drop_nulls() - elimina os valores nulos - há decisões de 1A que foram revistas, suspensas ou canceladas, resultando em valor nulo para as demais colunas;

3. filter() - filtra para as decisões tomadas após 2018 (antes disso os prazos entre 1A e PAGTO não eram homogêneos)

'''
visualiza_periodo_decisao_pagamento = penalidades_sc.select(pl.col(
                                            [
                                            'DT_PUBLICACAO_1A_FINAL',
                                            'DT_PAGTO_A_VISTA_ANS',
                                            'VL_PAGO_A_VISTA_ANS'
                                            ])
                                            ).drop_nulls(
                                                ).filter(
                                                    (pl.col('DT_PUBLICACAO_1A_FINAL') > date(2018,1,1))
                                                    ).sort('DT_PUBLICACAO_1A_FINAL', 
                                                       descending=True
                                                       ).show(
                                                            limit=None, 
                                                            decimal_separator=',', 
                                                            thousands_separator='.',
                                                            )

DT_PUBLICACAO_1A_FINAL,DT_PAGTO_A_VISTA_ANS,VL_PAGO_A_VISTA_ANS
date,date,i64
2025-01-20,2025-03-10,48.000
2025-01-13,2025-02-18,48.000
2024-12-26,2025-02-17,70.400
2024-12-20,2025-02-12,48.000
2024-12-10,2025-02-06,70.400
2024-12-02,2025-01-07,48.000
2024-11-26,2025-02-10,70.400
2024-10-25,2024-12-23,70.400
2024-10-25,2024-12-20,70.400
